Проверим метрики для прогнозов gpt

В chatgpt грузили обучающую выборку (со стратификацией по классам и обогащение малых классов всем доступными примерами - см. gpt_dataset.ipynb) на 50 тысяч примеров и использовали тест на 5000 примеров, в контекст gemini влезла только обучающая выборка на 15 тысяч примеров (тоже стратифицированная и обогащенная).
В YandexGPT lite станндартную модель для классификации загрузили поочередно ту же тестовую выборку на 5000 строк и отдельно еще использовали дообученную модель YandexGPT lite на ОШИБОЧНО дообученных 5000 примерах - поэтому в результатае такого дообучения получили отсутствие эффекта относительно станджартной модели ygpt.

In [1]:

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

y_true = pd.read_csv('/Users/roman_yakovlev/Documents/LemonPie/Сегментация_своя/gpt_classification/ver_1/y_test.csv', index_col=0)
y_pred_chatgpt = pd.read_csv('/Users/roman_yakovlev/Documents/LemonPie/Сегментация_своя/gpt_classification/ver_1/y_pred_chatgpt.csv',index_col=0)
y_pred_gemini = pd.read_csv('/Users/roman_yakovlev/Documents/LemonPie/Сегментация_своя/gpt_classification/ver_1/y_pred_gemini.csv',index_col=0)
y_pred_ygpt = pd.read_csv('/Users/roman_yakovlev/Documents/LemonPie/Сегментация_своя/gpt_classification/ver_1/y_pred_ygpt.csv',index_col=0)
y_pred_ygpt_ft = pd.read_csv('/Users/roman_yakovlev/Documents/LemonPie/Сегментация_своя/gpt_classification/ver_1/y_pred_ygpt_ft.csv',index_col=0)

In [ ]:
# gpt могут галлюцинировать, поэтому сверяем индексы у прогнозов и истинных значений
print(y_true.index.equals(y_pred_chatgpt.index))

# проверка, что множества индексов совпадают (порядок не важен)
print(set(y_true.index) == set(y_pred_chatgpt.index))

# вывод различий
print("Есть в y_true, но нет в y_pred:", y_true.index.difference(y_pred_chatgpt.index))
print("Есть в y_pred, но нет в y_true:", y_pred_chatgpt.index.difference(y_true.index))

True
True
Есть в y_true, но нет в y_pred: Index([], dtype='int64')
Есть в y_pred, но нет в y_true: Index([], dtype='int64')


In [3]:
# считаем метрики для прогнозв chatgpt

accuracy = accuracy_score(y_true, y_pred_chatgpt)

precision = precision_score(y_true, y_pred_chatgpt, average='weighted')
recall = recall_score(y_true, y_pred_chatgpt, average='weighted')
f1 = f1_score(y_true, y_pred_chatgpt, average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (weighted):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(y_true, y_pred_chatgpt))

Accuracy: 0.6602
Precision (weighted): 0.7783789525861354
Recall (weighted): 0.6602
F1-score (weighted): 0.653813605248847

Отчет по классам:
                                             precision    recall  f1-score   support

                   гранты субсидии конкурсы       0.80      1.00      0.89         4
 пожертвования от физических лиц (напрямую)       0.91      0.51      0.65      2989
пожертвования от юридических лиц (напрямую)       0.94      0.15      0.25       103
              пожертвования через платформы       0.51      0.97      0.67      1656
                            продажа товаров       0.89      0.62      0.73        55
                              продажа услуг       0.97      0.41      0.58        78
                 прочие недоходные операции       0.97      0.77      0.86        87
                          финансовые доходы       0.79      0.95      0.86        20
           членские и учредительские взносы       1.00      0.75      0.86         8

      

In [ ]:
# сверка индексов и выравнивание при необходимости
print(y_true.index.equals(y_pred_gemini.index))

# проверка, что множества индексов совпадают (порядок не важен)
print(set(y_true.index) == set(y_pred_gemini.index))

# вывод различий
print("Есть в y_true, но нет в y_pred:", y_true.index.difference(y_pred_gemini.index))
print("Есть в y_pred, но нет в y_true:", y_pred_gemini.index.difference(y_true.index))

False
True
Есть в y_true, но нет в y_pred: Index([], dtype='int64')
Есть в y_pred, но нет в y_true: Index([], dtype='int64')


In [5]:
y_pred_gemini = y_pred_gemini.reindex(y_true.index)

In [6]:
# считаем метрики для прогнозв gemini

accuracy = accuracy_score(y_true, y_pred_gemini)

precision = precision_score(y_true, y_pred_gemini, average='weighted')
recall = recall_score(y_true, y_pred_gemini, average='weighted')
f1 = f1_score(y_true, y_pred_gemini, average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (weighted):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(y_true, y_pred_gemini))

Accuracy: 0.5302
Precision (weighted): 0.46495971437412126
Recall (weighted): 0.5302
F1-score (weighted): 0.4845368412908898

Отчет по классам:
                                             precision    recall  f1-score   support

                   гранты субсидии конкурсы       1.00      1.00      1.00         4
 пожертвования от физических лиц (напрямую)       0.59      0.77      0.67      2989
пожертвования от юридических лиц (напрямую)       0.55      0.50      0.53       103
              пожертвования через платформы       0.19      0.10      0.13      1656
                            продажа товаров       0.63      0.22      0.32        55
                              продажа услуг       0.55      0.36      0.43        78
                 прочие недоходные операции       0.90      0.87      0.89        87
                          финансовые доходы       0.86      0.90      0.88        20
           членские и учредительские взносы       0.80      1.00      0.89         8

    

In [ ]:
# сверка индексов
print(y_true.index.equals(y_pred_ygpt.index))

# проверка, что множества индексов совпадают (порядок не важен)
print(set(y_true.index) == set(y_pred_ygpt.index))

# вывод различий
print("Есть в y_true, но нет в y_pred:", y_true.index.difference(y_pred_ygpt.index))
print("Есть в y_pred, но нет в y_true:", y_pred_ygpt.index.difference(y_true.index))

True
True
Есть в y_true, но нет в y_pred: Index([], dtype='int64')
Есть в y_pred, но нет в y_true: Index([], dtype='int64')


In [8]:
# считаем метрики для прогнозов YandexGPT

accuracy = accuracy_score(y_true, y_pred_ygpt)

precision = precision_score(y_true, y_pred_ygpt, average='weighted')
recall = recall_score(y_true, y_pred_ygpt, average='weighted')
f1 = f1_score(y_true, y_pred_ygpt, average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (weighted):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(y_true, y_pred_ygpt))

Accuracy: 0.473
Precision (weighted): 0.37000901305649275
Recall (weighted): 0.473
F1-score (weighted): 0.4038356210372617

Отчет по классам:
                                             precision    recall  f1-score   support

                   гранты субсидии конкурсы       0.80      1.00      0.89         4
 пожертвования от физических лиц (напрямую)       0.57      0.77      0.66      2989
пожертвования от юридических лиц (напрямую)       0.08      0.03      0.04       103
              пожертвования через платформы       0.02      0.00      0.00      1656
                            продажа товаров       0.67      0.11      0.19        55
                              продажа услуг       0.08      0.23      0.12        78
                 прочие недоходные операции       0.61      0.16      0.25        87
                          финансовые доходы       0.04      1.00      0.07        20
           членские и учредительские взносы       0.36      1.00      0.53         8

      

In [ ]:
# сверка индексов
print(y_true.index.equals(y_pred_ygpt_ft.index))

# проверка, что множества индексов совпадают (порядок не важен)
print(set(y_true.index) == set(y_pred_ygpt_ft.index))

# вывод различий
print("Есть в y_true, но нет в y_pred:", y_true.index.difference(y_pred_ygpt_ft.index))
print("Есть в y_pred, но нет в y_true:", y_pred_ygpt_ft.index.difference(y_true.index))

True
True
Есть в y_true, но нет в y_pred: Index([], dtype='int64')
Есть в y_pred, но нет в y_true: Index([], dtype='int64')


In [13]:
# считаем метрики для прогнозов YandexGPT дообученой на нашем датасете

accuracy = accuracy_score(y_true, y_pred_ygpt_ft)

precision = precision_score(y_true, y_pred_ygpt_ft, average='weighted')
recall = recall_score(y_true, y_pred_ygpt_ft, average='weighted')
f1 = f1_score(y_true, y_pred_ygpt_ft, average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (weighted):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(y_true, y_pred_ygpt_ft))

Accuracy: 0.4464
Precision (weighted): 0.33813895597515853
Recall (weighted): 0.4464
F1-score (weighted): 0.3828215111680662

Отчет по классам:
                                             precision    recall  f1-score   support

                   гранты субсидии конкурсы       0.00      0.00      0.00         4
 пожертвования от физических лиц (напрямую)       0.56      0.72      0.63      2989
пожертвования от юридических лиц (напрямую)       0.05      0.19      0.08       103
              пожертвования через платформы       0.00      0.00      0.00      1656
                            продажа товаров       0.00      0.00      0.00        55
                              продажа услуг       0.12      0.38      0.19        78
                 прочие недоходные операции       0.15      0.43      0.23        87
                          финансовые доходы       0.00      0.00      0.00        20
           членские и учредительские взносы       0.02      0.50      0.03         8

    

/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(re